# 투구 제구 성공 확률 모델 학습

이 노트북을 위에서 아래로 실행하면 시간순 OOF 검증, CatBoost RMSE/Logloss 블렌딩, 시간 OOF 기반 확률 보정, 최종 모델 저장까지 완료됩니다. 기본 설정은 전체 147만 행을 사용합니다.

In [1]:
# 최초 실행 환경에 CatBoost가 없으면 설치합니다.
import importlib.util
import subprocess
import sys

required_packages = {
    "catboost": "catboost==1.2.8",
    "sklearn": "scikit-learn==1.7.0",
    "pyarrow": "pyarrow==20.0.0",
}
missing_packages = [package for module, package in required_packages.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])

import gc
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.metrics import log_loss, mean_squared_error, roc_auc_score

from result import (
    ID_COL, TARGET_COL, apply_calibration, build_features,
    initial_feature_config, read_compact_csv, run_inference, save_json,
)

ROOT = Path.cwd().resolve()
if not (ROOT / "result.py").exists():
    raise FileNotFoundError("main.ipynb를 result.py가 있는 프로젝트 루트에서 실행해 주세요.")
DATA_DIR = ROOT / "open" / "data"
RUN_DIR = Path(os.environ.get("BASEBALL_RUN_DIR", ROOT)).resolve()
MODEL_DIR = RUN_DIR / "model"
ARTIFACT_DIR = RUN_DIR / "artifacts"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
VALID_YEARS = [2022, 2023, 2024]
ITERATIONS = int(os.environ.get("BASEBALL_ITERATIONS", "1200"))
DEPTH = int(os.environ.get("BASEBALL_DEPTH", "8"))
LEARNING_RATE = float(os.environ.get("BASEBALL_LEARNING_RATE", "0.04"))
EARLY_STOPPING_ROUNDS = int(os.environ.get("BASEBALL_EARLY_STOPPING", "120"))
MIN_FINAL_ITERATIONS = int(os.environ.get("BASEBALL_MIN_FINAL_ITERATIONS", "100"))
HALF_LIFE_YEARS = 3.0
FAST_MODE = os.environ.get("BASEBALL_FAST_MODE", "0") == "1"
FAST_ROWS_PER_SEASON = int(os.environ.get("BASEBALL_FAST_ROWS_PER_SEASON", "30000"))
print(f"project={ROOT}, run_dir={RUN_DIR}")


project=C:\Users\junhyun111\Desktop\baseball, run_dir=C:\Users\junhyun111\Desktop\baseball


## 1. 데이터 로드 및 피처 생성

리그 prior는 각 시즌보다 과거인 학습 라벨만 사용합니다. 평가 데이터 자체를 집계하는 피처는 만들지 않습니다.

In [2]:
started = time.time()
train = read_compact_csv(DATA_DIR / "train.csv", train=True)
if train[ID_COL].duplicated().any():
    raise ValueError("train row_id가 중복되었습니다.")
if not set(train[TARGET_COL].unique()).issubset({0, 1}):
    raise ValueError("target은 0/1이어야 합니다.")

if FAST_MODE:
    train = pd.concat([
        part.sample(min(len(part), FAST_ROWS_PER_SEASON), random_state=SEED)
        for _, part in train.groupby("season", sort=True)
    ], ignore_index=True)
    print("FAST_MODE: 산출물은 제출용으로 사용하면 안 됩니다.")

feature_config = initial_feature_config(train)
X = build_features(train, feature_config)
feature_config["feature_columns"] = list(X.columns)
feature_config["categorical_columns"] = [
    column for column in feature_config["categorical_columns"] if column in X.columns
]
metadata_columns = [ID_COL, "season", "game_type", "pitcher_id", "asof_pitcher_n", "asof_batter_n"]
oof_metadata = train[metadata_columns].copy()
y = train[TARGET_COL].to_numpy(dtype="int8")
seasons = train["season"].to_numpy(dtype="int16")
n_rows = len(train)
print(f"train={train.shape}, features={X.shape}, target_mean={y.mean():.6f}")
print(f"categorical={len(feature_config['categorical_columns'])}, elapsed={time.time()-started:.1f}s")
del train
gc.collect()


train=(1475092, 49), features=(1475092, 87), target_mean=0.523766
categorical=13, elapsed=5.2s


0

## 2. 시간순 OOF 학습

2019~2021→2022, 2019~2022→2023, 2019~2023→2024 순서로 검증합니다. 최근 시즌에는 더 높은 학습 가중치를 줍니다.

In [ ]:
def recency_weights(train_seasons, prediction_year, half_life=HALF_LIFE_YEARS):
    age = np.maximum(0, (prediction_year - 1) - np.asarray(train_seasons, dtype="float64"))
    weights = np.power(0.5, age / half_life)
    return (weights / weights.mean()).astype("float32")

def model_parameters(loss):
    params = dict(
        iterations=ITERATIONS, depth=DEPTH, learning_rate=LEARNING_RATE,
        l2_leaf_reg=8.0, random_strength=0.5, random_seed=SEED,
        allow_writing_files=False, thread_count=-1, verbose=100,
    )
    if loss == "RMSE":
        params.update(loss_function="RMSE", eval_metric="RMSE")
    else:
        params.update(loss_function="Logloss", eval_metric="BrierScore")
    return params

def brier(y_true, prediction):
    return float(mean_squared_error(y_true, np.clip(prediction, 1e-5, 1-1e-5)))

oof_rmse = np.full(n_rows, np.nan, dtype="float32")
oof_logloss = np.full(n_rows, np.nan, dtype="float32")
fold_rows = []
best_iterations = {"rmse": [], "logloss": []}
cat_columns = feature_config["categorical_columns"]

for valid_year in VALID_YEARS:
    fold_started = time.time()
    train_mask = seasons < valid_year
    valid_mask = seasons == valid_year
    if not train_mask.any() or not valid_mask.any():
        raise ValueError(f"{valid_year} fold 데이터가 없습니다.")
    X_train, X_valid = X.loc[train_mask], X.loc[valid_mask]
    y_train, y_valid = y[train_mask], y[valid_mask]
    weights = recency_weights(seasons[train_mask], valid_year)
    print(f"\nFold {valid_year}: train={len(y_train):,}, valid={len(y_valid):,}")

    rmse_model = CatBoostRegressor(**model_parameters("RMSE"))
    rmse_model.fit(
        X_train, y_train, cat_features=cat_columns, sample_weight=weights,
        eval_set=(X_valid, y_valid), early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        use_best_model=True,
    )
    pred_rmse = np.clip(rmse_model.predict(X_valid), 1e-5, 1-1e-5)
    oof_rmse[valid_mask] = pred_rmse.astype("float32")
    rmse_iteration = max(1, int(rmse_model.get_best_iteration()) + 1)
    best_iterations["rmse"].append(rmse_iteration)
    del rmse_model
    gc.collect()

    logloss_model = CatBoostClassifier(**model_parameters("Logloss"))
    logloss_model.fit(
        X_train, y_train, cat_features=cat_columns, sample_weight=weights,
        eval_set=(X_valid, y_valid), early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        use_best_model=True,
    )
    pred_logloss = np.clip(logloss_model.predict_proba(X_valid)[:, 1], 1e-5, 1-1e-5)
    oof_logloss[valid_mask] = pred_logloss.astype("float32")
    logloss_iteration = max(1, int(logloss_model.get_best_iteration()) + 1)
    best_iterations["logloss"].append(logloss_iteration)
    del logloss_model, X_train, X_valid, weights
    gc.collect()

    prior = float(y_train.mean())
    fold_rows.extend([
        {"season": valid_year, "model": "constant_train_prior", "brier": brier(y_valid, np.full(len(y_valid), prior)), "best_iteration": 0},
        {"season": valid_year, "model": "global_rmse", "brier": brier(y_valid, pred_rmse), "best_iteration": rmse_iteration},
        {"season": valid_year, "model": "global_logloss", "brier": brier(y_valid, pred_logloss), "best_iteration": logloss_iteration},
    ])
    print(f"Brier rmse={brier(y_valid, pred_rmse):.6f}, logloss={brier(y_valid, pred_logloss):.6f}, elapsed={time.time()-fold_started:.1f}s")

pd.DataFrame(fold_rows)



Fold 2022: train=728,588, valid=247,472
0:	learn: 0.4980348	test: 0.4988860	best: 0.4988860 (0)	total: 745ms	remaining: 14m 53s
100:	learn: 0.4927160	test: 0.4935682	best: 0.4935475 (91)	total: 50.3s	remaining: 9m 7s
200:	learn: 0.4919078	test: 0.4938830	best: 0.4935475 (91)	total: 1m 42s	remaining: 8m 30s
Stopped by overfitting detector  (120 iterations wait)

bestTest = 0.4935475323
bestIteration = 91

Shrink model to first 92 iterations.
0:	learn: 0.2495621	test: 0.2494865	best: 0.2494865 (0)	total: 598ms	remaining: 11m 57s


KeyboardInterrupt: 

: 

## 3. Walk-forward 블렌딩과 확률 보정

최종 제출용 계수는 모든 시간 OOF에서 학습하지만, 평가용 예측은 항상 해당 검증연도보다 앞선 OOF만 이용해 계수를 학습합니다.

In [ ]:
def optimal_rmse_weight(y_true, pred_rmse, pred_logloss):
    delta = np.asarray(pred_rmse, dtype="float64") - np.asarray(pred_logloss, dtype="float64")
    denominator = float(np.dot(delta, delta))
    if denominator <= 1e-15:
        return 0.5
    weight = float(np.dot(delta, np.asarray(y_true) - pred_logloss) / denominator)
    return float(np.clip(weight, 0.0, 1.0))

def fit_affine_calibrator(y_true, prediction):
    p = np.asarray(prediction, dtype="float64")
    target = np.asarray(y_true, dtype="float64")
    variance = float(np.mean((p - p.mean()) ** 2))
    covariance = float(np.mean((p - p.mean()) * (target - target.mean())))
    slope = covariance / (variance + 1e-8)
    slope = float(np.clip(slope, 0.5, 1.5))
    intercept = float(np.clip(target.mean() - slope * p.mean(), -0.10, 0.10))
    return slope, intercept

oof_mask = np.isfinite(oof_rmse) & np.isfinite(oof_logloss)
walk_blend = np.full(n_rows, np.nan, dtype="float32")
walk_affine_calibrated = np.full(n_rows, np.nan, dtype="float32")
walk_parameters = []

for valid_year in VALID_YEARS:
    current = oof_mask & (seasons == valid_year)
    history = oof_mask & (seasons < valid_year)
    if history.any():
        weight_rmse = optimal_rmse_weight(y[history], oof_rmse[history], oof_logloss[history])
    else:
        weight_rmse = 0.5
    current_blend = weight_rmse * oof_rmse[current] + (1-weight_rmse) * oof_logloss[current]
    walk_blend[current] = current_blend
    if history.any():
        history_blend = weight_rmse * oof_rmse[history] + (1-weight_rmse) * oof_logloss[history]
        slope, intercept = fit_affine_calibrator(y[history], history_blend)
    else:
        slope, intercept = 1.0, 0.0
    walk_affine_calibrated[current] = np.clip(slope * current_blend + intercept, 1e-5, 1-1e-5)
    walk_parameters.append({"season": valid_year, "rmse_weight": weight_rmse, "slope": slope, "intercept": intercept})

final_rmse_weight = optimal_rmse_weight(y[oof_mask], oof_rmse[oof_mask], oof_logloss[oof_mask])
final_blend = final_rmse_weight * oof_rmse[oof_mask] + (1-final_rmse_weight) * oof_logloss[oof_mask]
final_slope, final_intercept = fit_affine_calibrator(y[oof_mask], final_blend)
selection_years = VALID_YEARS[1:]
selection_weights = np.arange(1, len(selection_years) + 1, dtype="float64")
blend_scores = []
affine_scores = []
for year in selection_years:
    mask = oof_mask & (seasons == year)
    blend_scores.append(brier(y[mask], walk_blend[mask]))
    affine_scores.append(brier(y[mask], walk_affine_calibrated[mask]))
blend_selection_score = float(np.average(blend_scores, weights=selection_weights))
affine_selection_score = float(np.average(affine_scores, weights=selection_weights))
calibration_method = "affine" if affine_selection_score + 1e-7 < blend_selection_score else "identity"
walk_calibrated = walk_affine_calibrated.copy() if calibration_method == "affine" else walk_blend.copy()
if calibration_method == "affine":
    final_calibrated = np.clip(final_slope * final_blend + final_intercept, 1e-5, 1-1e-5)
else:
    final_calibrated = final_blend.copy()
print("walk-forward parameters")
display(pd.DataFrame(walk_parameters))
print(f"calibration selection: blend={blend_selection_score:.6f}, affine={affine_selection_score:.6f} -> {calibration_method}")
print(f"final rmse_weight={final_rmse_weight:.4f}, slope={final_slope:.4f}, intercept={final_intercept:.6f}")
for valid_year in VALID_YEARS:
    mask = oof_mask & (seasons == valid_year)
    print(valid_year, "blend", f"{brier(y[mask], walk_blend[mask]):.6f}", "calibrated", f"{brier(y[mask], walk_calibrated[mask]):.6f}")


## 4. OOF 산출물 저장 및 최종 모델 학습

In [ ]:
metric_rows = list(fold_rows)
for valid_year in VALID_YEARS:
    mask = oof_mask & (seasons == valid_year)
    for name, prediction in [("walk_forward_blend", walk_blend), ("walk_forward_calibrated", walk_calibrated)]:
        p = np.clip(prediction[mask], 1e-5, 1-1e-5)
        prior = float(y[seasons < valid_year].mean())
        baseline_brier = brier(y[mask], np.full(mask.sum(), prior))
        metric_rows.append({
            "season": valid_year, "model": name, "brier": brier(y[mask], p),
            "brier_skill_train_prior": 1 - brier(y[mask], p) / baseline_brier,
            "log_loss": float(log_loss(y[mask], p)),
            "roc_auc": float(roc_auc_score(y[mask], p)),
            "target_mean": float(y[mask].mean()), "prediction_mean": float(p.mean()),
            "best_iteration": 0,
        })
metrics = pd.DataFrame(metric_rows)
metrics.to_csv(ARTIFACT_DIR / "cv_metrics.csv", index=False, encoding="utf-8")

oof_frame = oof_metadata.loc[oof_mask].copy()
oof_frame[TARGET_COL] = y[oof_mask]
oof_frame["pred_rmse"] = oof_rmse[oof_mask]
oof_frame["pred_logloss"] = oof_logloss[oof_mask]
oof_frame["pred_blend_walk_forward"] = walk_blend[oof_mask]
oof_frame["pred_calibrated_walk_forward"] = walk_calibrated[oof_mask]
oof_frame["pred_blend_final_fit"] = final_blend.astype("float32")
oof_frame["pred_calibrated_final_fit"] = final_calibrated.astype("float32")
oof_frame.to_parquet(ARTIFACT_DIR / "oof_predictions.parquet", index=False)

rmse_iterations = max(MIN_FINAL_ITERATIONS, int(np.median(best_iterations["rmse"])))
logloss_iterations = max(MIN_FINAL_ITERATIONS, int(np.median(best_iterations["logloss"])))
ensemble_config = {
    "version": 1, "rmse_weight": final_rmse_weight,
    "logloss_weight": 1-final_rmse_weight,
    "rmse_iterations": rmse_iterations, "logloss_iterations": logloss_iterations,
}
calibration_config = {"version": 1, "method": calibration_method, "slope": final_slope, "intercept": final_intercept, "trained_on_seasons": VALID_YEARS, "walk_forward_blend_brier": blend_selection_score, "walk_forward_affine_brier": affine_selection_score}
save_json(MODEL_DIR / "feature_config.json", feature_config)
save_json(MODEL_DIR / "ensemble_config.json", ensemble_config)
save_json(MODEL_DIR / "calibration_config.json", calibration_config)
print(f"final iterations: rmse={rmse_iterations}, logloss={logloss_iterations}")


In [ ]:
final_weights = recency_weights(seasons, int(seasons.max()) + 1)
final_rmse_params = model_parameters("RMSE")
final_rmse_params.update(iterations=rmse_iterations, verbose=100)
final_rmse_model = CatBoostRegressor(**final_rmse_params)
final_rmse_model.fit(X, y, cat_features=cat_columns, sample_weight=final_weights)
final_rmse_model.save_model(str(MODEL_DIR / "global_rmse.cbm"))
rmse_importance = final_rmse_model.get_feature_importance()
del final_rmse_model
gc.collect()

final_logloss_params = model_parameters("Logloss")
final_logloss_params.update(iterations=logloss_iterations, verbose=100)
final_logloss_model = CatBoostClassifier(**final_logloss_params)
final_logloss_model.fit(X, y, cat_features=cat_columns, sample_weight=final_weights)
final_logloss_model.save_model(str(MODEL_DIR / "global_logloss.cbm"))
logloss_importance = final_logloss_model.get_feature_importance()
del final_logloss_model
gc.collect()

importance = pd.DataFrame({"feature": X.columns, "rmse_importance": rmse_importance, "logloss_importance": logloss_importance})
importance["mean_importance"] = importance[["rmse_importance", "logloss_importance"]].mean(axis=1)
importance.sort_values("mean_importance", ascending=False).to_csv(ARTIFACT_DIR / "feature_importance.csv", index=False, encoding="utf-8")
summary = {
    "fast_mode": FAST_MODE, "rows": n_rows, "features": X.shape[1],
    "valid_years": VALID_YEARS, "half_life_years": HALF_LIFE_YEARS,
    "best_iterations": best_iterations, "ensemble": ensemble_config,
    "calibration": calibration_config, "elapsed_seconds": time.time()-started,
}
save_json(ARTIFACT_DIR / "training_summary.json", summary)
print(f"학습 완료: {time.time()-started:.1f}s")
display(importance.sort_values("mean_importance", ascending=False).head(20))


## 5. 공개 샘플 추론 점검

실제 평가는 `result.py`가 동일한 방식으로 수행합니다.

In [ ]:
submission_path = run_inference(ROOT, data_dir=str(DATA_DIR), model_dir=str(MODEL_DIR), output_dir=str(RUN_DIR / "output"))
display(pd.read_csv(submission_path))
print("다음으로 evaluation.ipynb를 실행해 OOF 평가를 확인하세요.")
